In [1]:
pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
Design Document Generator using OpenAI + Python
================================================
Generates a professional technical design document (Word .docx) from a subject/topic
using the OpenAI API. Saves output as a formatted Word document.

Requirements:
    pip install openai python-docx

Usage:
    python generate_design_doc.py --subject "E-Commerce Recommendation Engine"
    python generate_design_doc.py --subject "Microservices Auth System" --type HLD
    python generate_design_doc.py  # interactive prompt
"""

import argparse
import json
import os
import subprocess
import sys
import tempfile
from datetime import date



In [3]:
# ── OpenAI client setup ──────────────────────────────────────────────────────
try:
    from openai import OpenAI
except ImportError:
    print("❌  openai package not found. Run: pip install openai")
    sys.exit(1)

In [4]:
import json
from datetime import date
from openai import OpenAI

#SYSTEM_PROMPT = """You are a senior software architect who writes clear, structured,
#professional technical design documents. Return ONLY valid JSON — no markdown fences,
#no commentary before or after. Follow the exact schema provided."""


SYSTEM_PROMPT = """You are a senior software architect with extensive experience in designing, documenting, and communicating complex software systems. Your role is to generate clear, structured, professional technical design documents that adhere to industry standards and best practices. You must always return ONLY valid JSON output — no markdown fences, no commentary before or after, no explanatory text outside the JSON. The JSON must strictly follow the schema provided by the user, with all required fields present and populated. Your responses must be deterministic, consistent, and free of ambiguity.

Your writing style should reflect the qualities of a seasoned architect: precision, clarity, and depth. Each section of the JSON must be filled with meaningful, contextually relevant content that demonstrates a deep understanding of software architecture principles. Avoid generic placeholders unless explicitly instructed; instead, provide detailed, realistic examples that could plausibly exist in a professional design document. Your tone should be authoritative yet accessible, ensuring that both technical and non-technical stakeholders can understand the document.

When writing the executive summary, provide a concise overview of the system’s purpose, scope, and expected outcomes. This should be no longer than 2–3 sentences but must capture the essence of the design. Objectives should be actionable, measurable, and aligned with business goals. Scope must clearly delineate what is included and excluded, preventing misunderstandings later in the project lifecycle.

The architecture overview should describe the system at a high level, including its major layers, components, and interactions. Use terminology consistent with modern architectural practices, such as microservices, event-driven design, layered architecture, or cloud-native deployment. Components must be described in detail, including their responsibilities, technologies, interfaces, dependencies, and scalability strategies. Each component should feel like a real part of a system, not a placeholder.

Data design must include an overview of the data model, key entities, relationships, indexing strategies, and retention policies. Show awareness of normalization, denormalization, and trade-offs between relational and NoSQL approaches. Include considerations for compliance, privacy, and performance. Key entities should be named realistically, such as User, Order, Product, Payment, etc., and relationships should be described in terms of cardinality and constraints.

API design must specify the style (REST, GraphQL, gRPC, etc.), endpoints, methods, paths, descriptions, authentication requirements, rate limits, and error codes. Endpoints should be realistic and aligned with the system’s domain. For example, a food delivery system might have endpoints for /orders, /restaurants, /users, etc. Include both success and failure scenarios, demonstrating robustness.

Security must cover authentication, authorization, encryption, compliance, monitoring, and logging. Mention industry-standard practices such as OAuth2, JWT, TLS, RBAC, and audit trails. Consider regulatory frameworks such as GDPR, HIPAA, or ISO 27001 if relevant. Security considerations should be practical and enforceable, not vague.

Scalability must describe strategies for handling growth in users, data, and traffic. Include horizontal scaling, load balancing, caching, asynchronous processing, and disaster recovery. Mention cloud-native approaches such as Kubernetes, autoscaling groups, and multi-region deployments. Show awareness of bottlenecks and mitigation strategies.

Non-functional requirements must cover performance, availability, reliability, maintainability, security, usability, and portability. Each requirement should be specific, measurable, and testable. For example, “Average response time < 200ms under 1000 concurrent users” is better than “System should be fast.” Availability should be expressed in terms of SLA percentages. Reliability should mention failover and redundancy. Maintainability should include code quality, documentation, and test coverage. Security should include encryption and compliance. Usability should reference accessibility standards. Portability should mention deployment across multiple cloud providers.

Risks must include realistic threats to the project, such as data breaches, downtime, scope creep, or vendor lock-in. Each risk must have a mitigation strategy, severity, and likelihood. Severity should be High, Medium, or Low. Likelihood should also be High, Medium, or Low. Mitigation strategies should be actionable, such as “Implement encryption at rest” or “Use redundant infrastructure.”

Timeline must break the project into phases such as Discovery, Development, Testing, and Deployment. Each phase must include duration, deliverables, dependencies, and milestones. Durations should be realistic (e.g., 2 weeks for discovery, 6 weeks for development). Deliverables should be tangible, such as “Requirements document,” “Working prototype,” or “Production deployment.” Dependencies should show logical sequencing. Milestones should be specific, such as “API implementation complete” or “User acceptance testing passed.”

Project deliverables must be listed explicitly as a separate section in the JSON. These deliverables represent the tangible outputs of the project, such as documentation, prototypes, deployed systems, or training materials. Each deliverable should be clear, measurable, and aligned with the objectives.

Project acceptance criteria must also be included as a separate section in the JSON. Acceptance criteria define the conditions under which the project will be considered complete and successful. They should be specific, measurable, and agreed upon by stakeholders. Examples include “System passes all functional and non-functional tests,” “Stakeholder sign-off received,” or “Deployment completed with zero critical defects.” Include a sign-off authority field to indicate who is responsible for final acceptance.

The conclusion must summarize the document, reinforce the importance of the design, and outline next steps. It should be professional, concise, and forward-looking. For example: “This document provides a comprehensive design for the system. Next steps include stakeholder review, prototype validation, and readiness assessment for deployment.”

Throughout the JSON, ensure consistency of terminology, formatting, and style. Use realistic values rather than placeholders whenever possible. Avoid contradictions or omissions. Every required field must be present and populated. Do not add extra fields unless explicitly instructed. Do not output explanatory text outside the JSON. Do not wrap the JSON in markdown fences. Do not include comments. Return ONLY valid JSON.

Your role is not just to fill in fields but to produce a document that could be used in a real-world project. Think like an architect presenting to a team of engineers, product managers, and executives. Provide enough detail to guide implementation while remaining high-level enough to avoid overwhelming the reader. Balance technical depth with clarity. Ensure that the document is professional, polished, and ready for stakeholder review.

In summary: You are a senior software architect. You write clear, structured, professional technical design documents. You return ONLY valid JSON. You follow the exact schema provided. You enrich each section with realistic, detailed, contextually appropriate content. You ensure consistency, clarity, and professionalism. You avoid commentary outside the JSON. You produce documents that are authoritative, actionable, and aligned with industry best practices. You must include both project deliverables and project acceptance criteria in the JSON schema.

"""

DOC_TYPES = {
    "HLD": "High-Level Design (HLD)",
    "LLD": "Low-Level Design (LLD)",
    "SDD": "Software Design Document (SDD)",
    "TDD": "Technical Design Document (TDD)",
    "PRD": "Product Requirements Document (PRD)",
}

def build_user_prompt(subject: str, doc_type: str) -> str:
    return f"""Create a detailed {DOC_TYPES.get(doc_type, doc_type)} for: "{subject}"

Return EXACTLY this JSON structure (all fields required):
{{
  "title": "<document title>",
  "version": "1.0",
  "date": "{date.today().isoformat()}",
  "authors": ["Your Name"],
  "doc_type": "{DOC_TYPES.get(doc_type, doc_type)}",
  "executive_summary": "<2-3 sentence overview>",
  "objectives": ["<objective 1>", "<objective 2>", "<objective 3>"],
  "scope": {{
    "in_scope": ["<item 1>", "<item 2>", "<item 3>"],
    "out_of_scope": ["<item 1>", "<item 2>"]
  }},
  "architecture_overview": "<paragraph describing the system architecture>",
  "components": [
    {{
      "name": "<Component Name>",
      "description": "<what it does>",
      "responsibilities": ["<resp 1>", "<resp 2>"],
      "technology": "<tech stack>"
    }}
  ],
  "data_design": {{
    "overview": "<data model overview>",
    "key_entities": ["<Entity1>", "<Entity2>", "<Entity3>"],
    "storage": "<database/storage technology>"
  }},
  "api_design": {{
    "style": "<REST/GraphQL/gRPC/etc>",
    "endpoints": [
      {{"method": "GET",  "path": "/example",        "description": "<what it does>"}},
      {{"method": "POST", "path": "/example",        "description": "<what it does>"}},
      {{"method": "PUT",  "path": "/example/{{id}}", "description": "<what it does>"}}
    ]
  }},
  "security": {{
    "authentication": "<auth mechanism>",
    "authorization": "<authz approach>",
    "considerations": ["<security point 1>", "<security point 2>"]
  }},
  "scalability": "<paragraph on scaling strategy>",
  "non_functional_requirements": [
    {{"category": "Performance",    "requirement": "<requirement>"}},
    {{"category": "Availability",   "requirement": "<requirement>"}},
    {{"category": "Reliability",    "requirement": "<requirement>"}},
    {{"category": "Maintainability","requirement": "<requirement>"}}
  ],
  "risks": [
    {{"risk": "<risk description>", "mitigation": "<mitigation strategy>"}},
    {{"risk": "<risk description>", "mitigation": "<mitigation strategy>"}}
  ],
  "timeline": [
    {{"phase": "Phase 1 – Discovery",     "duration": "2 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 2 – Development",   "duration": "6 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 3 – Testing",       "duration": "2 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 4 – Deployment",    "duration": "1 week",  "deliverable": "<deliverable>"}}
  ],
   "deliverables": [
    "<deliverable 1>",
    "<deliverable 2>",
    "<deliverable 3>"
  ],
  "project_acceptance": {{
    "criteria": ["<acceptance criterion 1>", "<acceptance criterion 2>", "<acceptance criterion 3>"],
    "sign_off": "<stakeholder or authority responsible for acceptance>"
  }},
  "conclusion": "<closing paragraph summarising the document>"
}}"""

def generate_design_doc(subject: str, doc_type: str = "HLD") -> dict:
    client = OpenAI()
    user_prompt = build_user_prompt(subject, doc_type)

    completion = client.chat.completions.create(
        model="gpt-4.1",  # or whichever model you prefer
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,  # keep deterministic for JSON
    )

    raw = completion.choices[0].message.content.strip()
    return json.loads(raw)




In [5]:
import json
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

# --- colour palette ---
BLUE   = "1F4E79"
LBLUE  = "2E75B6"
WHITE  = "FFFFFF"
LGRAY  = "F2F2F2"
DTEXT  = "2C2C2C"
MTEXT  = "444444"

def h1(doc, text):
    p = doc.add_paragraph(text, style="Heading 1")
    run = p.runs[0]
    run.bold = True
    run.font.size = Pt(28)
    run.font.color.rgb = RGBColor.from_string(BLUE)
    return p

def h2(doc, text):
    p = doc.add_paragraph(text, style="Heading 2")
    run = p.runs[0]
    run.bold = True
    run.font.size = Pt(24)
    run.font.color.rgb = RGBColor.from_string(LBLUE)
    return p

def body(doc, text):
    p = doc.add_paragraph(text)
    run = p.runs[0]
    run.font.size = Pt(20)
    run.font.color.rgb = RGBColor.from_string(MTEXT)
    return p

def bullet(doc, text):
    p = doc.add_paragraph(text, style="List Bullet")
    run = p.runs[0]
    run.font.size = Pt(20)
    run.font.color.rgb = RGBColor.from_string(MTEXT)
    return p

def space(doc):
    doc.add_paragraph("")

def build_docx_from_json(data: dict, out_path: str):
    doc = Document()

    # Cover page
    p1 = doc.add_paragraph(f"◆  {data['doc_type']}  ◆")
    p1.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run1 = p1.runs[0]
    run1.bold = True
    run1.font.size = Pt(28)
    run1.font.color.rgb = RGBColor.from_string(LBLUE)

    p2 = doc.add_paragraph(data["title"])
    p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run2 = p2.runs[0]
    run2.bold = True
    run2.font.size = Pt(56)
    run2.font.color.rgb = RGBColor.from_string(BLUE)

    doc.add_paragraph(f"Version {data['version']} | {data['date']}").alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph("Author(s): " + ", ".join(data["authors"])).alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_page_break()

    # Executive Summary
    h1(doc, "1. Executive Summary")
    body(doc, data["executive_summary"])
    space(doc)

    # Objectives
    h1(doc, "2. Objectives")
    for o in data["objectives"]:
        bullet(doc, o)
    space(doc)

    # Scope
    h1(doc, "3. Scope")
    h2(doc, "3.1 In-Scope")
    for s in data["scope"]["in_scope"]:
        bullet(doc, s)
    h2(doc, "3.2 Out-of-Scope")
    for s in data["scope"]["out_of_scope"]:
        bullet(doc, s)
    space(doc)

    # Architecture Overview
    h1(doc, "4. Architecture Overview")
    body(doc, data["architecture_overview"])
    space(doc)

    # Components
    h1(doc, "5. System Components")
    for i, c in enumerate(data["components"], start=1):
        h2(doc, f"5.{i} {c['name']}")
        body(doc, "Description: " + c["description"])
        body(doc, "Technology: " + c["technology"])
        body(doc, "Responsibilities:")
        for r in c["responsibilities"]:
            bullet(doc, r)
        space(doc)

    # Data Design
    h1(doc, "6. Data Design")
    body(doc, data["data_design"]["overview"])
    body(doc, "Storage: " + data["data_design"]["storage"])
    h2(doc, "Key Entities")
    for e in data["data_design"]["key_entities"]:
        bullet(doc, e)
    space(doc)

    # API Design
    h1(doc, "7. API Design")
    body(doc, "Style: " + data["api_design"]["style"])
    table = doc.add_table(rows=1, cols=3)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text, hdr_cells[1].text, hdr_cells[2].text = "Method", "Endpoint", "Description"
    for ep in data["api_design"]["endpoints"]:
        row_cells = table.add_row().cells
        row_cells[0].text, row_cells[1].text, row_cells[2].text = ep["method"], ep["path"], ep["description"]
    space(doc)

    # Security
    h1(doc, "8. Security")
    body(doc, "Authentication: " + data["security"]["authentication"])
    body(doc, "Authorization: " + data["security"]["authorization"])
    for s in data["security"]["considerations"]:
        bullet(doc, s)
    space(doc)

    # Scalability
    h1(doc, "9. Scalability")
    body(doc, data["scalability"])
    space(doc)

    # Non-Functional Requirements
    h1(doc, "10. Non-Functional Requirements")
    table = doc.add_table(rows=1, cols=2)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text, hdr_cells[1].text = "Category", "Requirement"
    for nfr in data["non_functional_requirements"]:
        row_cells = table.add_row().cells
        row_cells[0].text, row_cells[1].text = nfr["category"], nfr["requirement"]
    space(doc)

    # Risks
    h1(doc, "11. Risks & Mitigations")
    table = doc.add_table(rows=1, cols=2)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text, hdr_cells[1].text = "Risk", "Mitigation"
    for r in data["risks"]:
        row_cells = table.add_row().cells
        row_cells[0].text, row_cells[1].text = r["risk"], r["mitigation"]
    space(doc)
   
    # --- Project Acceptance Criteria ---
    h1(doc, "12. Project Acceptance Criteria")
    table = doc.add_table(rows=1, cols=2)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text, hdr_cells[1].text = "Criteria", "Sign-off"
    
    # Expecting data["project_acceptance"] to be a dict with "criteria" list and "sign_off" string
    for criterion in data["project_acceptance"]["criteria"]:
        row_cells = table.add_row().cells
        row_cells[0].text = criterion
        row_cells[1].text = data["project_acceptance"]["sign_off"]
    
    space(doc)
    
    # --- Project Deliverables ---
    h1(doc, "13. Project Deliverables")
    table = doc.add_table(rows=1, cols=1)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = "Deliverables"
    
    # Expecting data["deliverables"] to be a list of strings
    for deliverable in data["deliverables"]:
        row_cells = table.add_row().cells
        row_cells[0].text = deliverable
    
    space(doc)

    # Timeline
    h1(doc, "14. Project Timeline")
    table = doc.add_table(rows=1, cols=3)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text, hdr_cells[1].text, hdr_cells[2].text = "Phase", "Duration", "Deliverable"
    for t in data["timeline"]:
        row_cells = table.add_row().cells
        row_cells[0].text, row_cells[1].text, row_cells[2].text = t["phase"], t["duration"], t["deliverable"]
    space(doc)

    # Conclusion
    h1(doc, "13. Conclusion")
    body(doc, data["conclusion"])

    doc.save(out_path)


# Example usage
if __name__ == "__main__":
   projname = input("Enter Project Name: ")
   for code, full in DOC_TYPES.items():
    print(f"{code}: {full}")
    doc = generate_design_doc(projname, code)
    print(json.dumps(doc, indent=2))
    # json_str = <LLM output string>
    data = json.loads(json.dumps(doc, indent=2))
    build_docx_from_json(data, projname+"_design_doc_"+code+".docx")


Enter Project Name:  Food Delivery


HLD: High-Level Design (HLD)
{
  "title": "High-Level Design for Food Delivery Platform",
  "version": "1.0",
  "date": "2026-04-04",
  "authors": [
    "Your Name"
  ],
  "doc_type": "High-Level Design (HLD)",
  "executive_summary": "This document outlines the high-level design for a scalable, secure, and user-friendly food delivery platform connecting customers, restaurants, and delivery partners. The system aims to streamline order placement, real-time tracking, and payment processing while ensuring high availability and data security. The design leverages modern cloud-native architecture to support rapid growth and evolving business needs.",
  "objectives": [
    "Enable seamless food ordering and delivery experience for customers.",
    "Provide real-time order tracking and status updates.",
    "Ensure secure, reliable, and scalable platform operations."
  ],
  "scope": {
    "in_scope": [
      "Customer registration and authentication",
      "Restaurant onboarding and menu man